# 동적 웹페이지 크롤링 : (5) Naver News

네이버에서 검색어를 입력하고 뉴스 탭으로 이동한 뒤, **현재 검색 결과 페이지**의 메인 기사를 수집한다.

```text
네이버 접속
→ 검색어 입력
→ 뉴스 탭 이동
→ 현재 페이지 기사 추출
→ 이미지 다운로드
→ 메타데이터 CSV 저장
```

**수집 항목**
- 언론사
- 기사 제목
- 기사 요약
- 기사 URL
- 이미지 URL
- 검색어
- 현재 페이지에서의 기사 순번
- 검색 결과 페이지 URL
- 수집 시각
- 로컬 이미지 파일 경로

> 실제 사이트는 DOM 구조와 운영 정책이 변경될 수 있으므로 실행 전 이용약관, robots.txt, 요청 범위를 확인한다.


# 라이브러리

In [17]:
import json
import re
from datetime import datetime
from pathlib import Path
from urllib.parse import parse_qs, unquote, urlparse

import pandas as pd
import requests
from requests.adapters import HTTPAdapter
from selenium import webdriver
from selenium.common.exceptions import NoSuchElementException, StaleElementReferenceException
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.common.keys import Keys
from selenium.webdriver.remote.webelement import WebElement
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.support.ui import WebDriverWait
from urllib3.util.retry import Retry

# 기본 설정

In [18]:
TARGET_URL = 'https://www.naver.com/'
SEARCH_KEYWORD = 'AI'

WAIT_TIMEOUT = 10

## False : 브라우저 화면 표시
## True : 브라우저 화면을 표시하지 않고 실행
HEADLESS = False

CONNECT_TIMEOUT = 10
READ_TIMEOUT = 30

PROJECT_DIR = Path.cwd().resolve().parents[1]
OUTPUT_DIR = PROJECT_DIR / 'data' / 'dynamic' / 'naver'

TITLE_SELECTOR = (
    'a[data-heatmap-target=".tit"] '
    '> span.sds-comps-text-type-headline1'
)
PRESS_SELECTOR = (
    '.sds-comps-profile-info-title-text '
    'a[href*="media.naver.com/press/"] '
    'span.sds-comps-text'
)
SUMMARY_SELECTOR = (
    'a[data-heatmap-target=".body"] '
    'span.sds-comps-text-type-body1'
)
IMAGE_SELECTOR = 'a[data-heatmap-target=".img"] img'
NEWS_CARD_XPATH = './ancestor::div[.//*[@data-sds-comp="Profile"]][1]'

# Chrome WebDriver 생성

`HEADLESS`는 호출할 때 명시적으로 전달한다.


In [19]:
def create_driver(headless: bool) -> webdriver.Chrome:
    """Chrome WebDriver를 생성하여 반환한다."""

    options = Options()

    if headless:
        options.add_argument('--headless=new')

    options.add_argument('--start-maximized')

    return webdriver.Chrome(options=options)

# 네이버 뉴스 검색

In [20]:
def open_naver_news_search(
    driver: webdriver.Chrome,
    search_keyword: str,
    wait_timeout: int,
) -> str:
    """
    네이버에서 검색어를 입력하고 뉴스 탭으로 이동한다.

    Returns:
        뉴스 검색 결과 페이지 URL
    """

    wait = WebDriverWait(driver, wait_timeout)

    driver.get(TARGET_URL)

    query = wait.until(
        EC.element_to_be_clickable((By.ID, 'query'))
    )
    query.clear()
    query.send_keys(search_keyword)
    query.send_keys(Keys.ENTER)

    news_tab = wait.until(
        EC.element_to_be_clickable(
            (
                By.XPATH,
                "//div[@id='lnb']//a[@role='tab' and normalize-space()='뉴스']",
            )
        )
    )
    news_tab.click()

    wait.until(
        EC.presence_of_element_located(
            (By.CSS_SELECTOR, TITLE_SELECTOR)
        )
    )

    return driver.current_url

# 파일 및 폴더 이름 정리

In [21]:
def sanitize_name(value: str, max_length: int = 80) -> str:
    """폴더명 또는 파일명으로 사용할 문자열을 정리한다."""

    value = re.sub(r'[\\/:*?"<>|]', '_', value)
    value = re.sub(r'\s+', ' ', value).strip()
    value = value.rstrip('. ')

    return (value or 'untitled')[:max_length]


def create_batch_directory(
    output_dir: Path,
    search_keyword: str,
    collected_at: datetime,
) -> tuple[Path, Path]:
    """검색어와 수집 시각을 기준으로 배치 폴더와 이미지 폴더를 생성한다."""

    keyword_name = sanitize_name(search_keyword)
    batch_name = collected_at.strftime('%Y%m%d_%H%M%S')

    batch_dir = output_dir / keyword_name / batch_name
    image_dir = batch_dir / 'images'

    image_dir.mkdir(parents=True, exist_ok=True)

    return batch_dir, image_dir

# 뉴스 한 건 추출

자동 생성된 해시 클래스보다 현재 HTML에서 의미가 명확한 속성과 클래스를 사용한다.

- 제목 : `data-heatmap-target=".tit"`
- 요약 : `data-heatmap-target=".body"`
- 이미지 : `data-heatmap-target=".img"`
- 언론사 : `media.naver.com/press/`


In [22]:
def get_original_image_url(image_url: str) -> str:
    """
    네이버 검색 이미지 프록시 URL에 원본 src 파라미터가 있으면 반환한다.

    원본 URL을 찾지 못하면 입력받은 이미지 URL을 그대로 반환한다.
    """

    if not image_url:
        return ''

    parsed = urlparse(image_url)
    source_values = parse_qs(parsed.query).get('src')

    if not source_values:
        return image_url

    return unquote(source_values[0])

In [23]:
def parse_news_item(
    title_element: WebElement,
    rank: int,
    search_keyword: str,
    source_url: str,
    collected_at: datetime,
) -> dict[str, str | int]:
    """메인 기사 제목 요소를 기준으로 뉴스 한 건의 정보를 추출한다."""

    news_card = title_element.find_element(
        By.XPATH,
        NEWS_CARD_XPATH,
    )

    press = news_card.find_element(
        By.CSS_SELECTOR,
        PRESS_SELECTOR,
    ).text.strip()

    title = title_element.text.strip()

    title_link = title_element.find_element(By.XPATH, '..')
    article_url = title_link.get_attribute('href') or ''

    summary = news_card.find_element(
        By.CSS_SELECTOR,
        SUMMARY_SELECTOR,
    ).text.strip()

    image_elements = news_card.find_elements(
        By.CSS_SELECTOR,
        IMAGE_SELECTOR,
    )

    image_url = ''

    if image_elements:
        image_url = (
            image_elements[0].get_property('currentSrc')
            or image_elements[0].get_attribute('src')
            or ''
        )

    return {
        'rank': rank,
        'search_keyword': search_keyword,
        'press': press,
        'title': title,
        'summary': summary,
        'article_url': article_url,
        'image_url': image_url,
        'image_source_url': get_original_image_url(image_url),
        'source_site': 'Naver News Search',
        'source_url': source_url,
        'collected_at': collected_at.isoformat(timespec='seconds'),
    }

# 현재 페이지 전체 기사 추출

In [24]:
def collect_current_news_page(
    driver: webdriver.Chrome,
    search_keyword: str,
    collected_at: datetime,
    wait_timeout: int,
) -> pd.DataFrame:
    """현재 뉴스 검색 결과 페이지의 메인 기사들을 수집한다."""

    wait = WebDriverWait(driver, wait_timeout)

    title_elements = wait.until(
        EC.presence_of_all_elements_located(
            (By.CSS_SELECTOR, TITLE_SELECTOR)
        )
    )

    source_url = driver.current_url
    news_records = []
    seen_article_urls = set()

    for rank, title_element in enumerate(title_elements, start=1):
        try:
            news = parse_news_item(
                title_element=title_element,
                rank=rank,
                search_keyword=search_keyword,
                source_url=source_url,
                collected_at=collected_at,
            )

        except (NoSuchElementException, StaleElementReferenceException) as error:
            print(f'[{rank:03d}] 기사 추출 실패 : {error.__class__.__name__}')
            continue

        article_url = news['article_url']

        if not article_url or article_url in seen_article_urls:
            continue

        seen_article_urls.add(article_url)
        news_records.append(news)

    news_df = pd.DataFrame(news_records)

    if news_df.empty:
        raise ValueError('현재 페이지에서 수집된 뉴스 기사가 없습니다.')

    if news_df['article_url'].duplicated().any():
        raise ValueError('중복된 기사 URL이 존재합니다.')

    print(f'현재 페이지 수집 기사 수 : {len(news_df)}')

    return news_df

# 이미지 다운로드용 HTTP Session

일시적인 네트워크 오류와 서버 오류에 대비해 retry/backoff를 적용한다.


In [25]:
def create_http_session() -> requests.Session:
    """이미지 다운로드에 사용할 HTTP Session을 생성한다."""

    retry = Retry(
        total=3,
        connect=3,
        read=3,
        status=3,
        backoff_factor=1.0,
        status_forcelist=(429, 500, 502, 503, 504),
        allowed_methods=frozenset({'GET'}),
        respect_retry_after_header=True,
    )

    session = requests.Session()
    session.mount('https://', HTTPAdapter(max_retries=retry))
    session.headers.update({
        'User-Agent': 'EducationalDataCollector/1.0',
    })

    return session

# 이미지 저장

In [26]:
def get_image_extension(response: requests.Response) -> str:
    """응답 Content-Type을 기준으로 이미지 확장자를 반환한다."""

    content_type = response.headers.get('Content-Type', '').split(';')[0].lower()

    extension_map = {
        'image/jpeg': '.jpg',
        'image/png': '.png',
        'image/webp': '.webp',
        'image/gif': '.gif',
    }

    return extension_map.get(content_type, '.jpg')


def download_image(
    session: requests.Session,
    image_url: str,
    image_dir: Path,
    rank: int,
    title: str,
) -> Path:
    """이미지를 요청하여 임시 파일을 거친 뒤 최종 파일로 저장한다."""

    response = session.get(
        image_url,
        timeout=(CONNECT_TIMEOUT, READ_TIMEOUT),
    )
    response.raise_for_status()

    content_type = response.headers.get('Content-Type', '').lower()

    if not content_type.startswith('image/'):
        raise ValueError(f'이미지 응답이 아닙니다. Content-Type={content_type}')

    extension = get_image_extension(response)
    safe_title = sanitize_name(title)

    image_file = image_dir / f'{rank:03d}_{safe_title}{extension}'
    temp_file = image_file.with_suffix(image_file.suffix + '.part')

    temp_file.write_bytes(response.content)
    temp_file.replace(image_file)

    return image_file

In [27]:
def download_news_images(
    news_df: pd.DataFrame,
    image_dir: Path,
    project_dir: Path,
) -> pd.DataFrame:
    """뉴스 DataFrame의 이미지들을 저장하고 로컬 파일 정보를 추가한다."""

    result_df = news_df.copy()

    result_df['image_file'] = ''
    result_df['image_downloaded'] = False

    session = create_http_session()

    try:
        for index, row in result_df.iterrows():
            image_url = row['image_url']

            if not image_url:
                continue

            try:
                image_file = download_image(
                    session=session,
                    image_url=image_url,
                    image_dir=image_dir,
                    rank=int(row['rank']),
                    title=row['title'],
                )

            except (requests.exceptions.RequestException, ValueError) as error:
                print(
                    f'[{int(row["rank"]):03d}] 이미지 저장 실패 '
                    f': {error.__class__.__name__}'
                )
                continue

            result_df.at[index, 'image_file'] = str(
                image_file.relative_to(project_dir)
            )
            result_df.at[index, 'image_downloaded'] = True

    finally:
        session.close()

    return result_df

# CSV 저장 및 검증

In [28]:
NEWS_COLUMNS = [
    'rank',
    'search_keyword',
    'press',
    'title',
    'summary',
    'article_url',
    'image_url',
    'image_source_url',
    'image_file',
    'image_downloaded',
    'source_site',
    'source_url',
    'collected_at',
]


def validate_news_dataframe(news_df: pd.DataFrame) -> None:
    """CSV 저장 전에 필수 컬럼, 결측값, 중복 URL을 검증한다."""

    missing_columns = set(NEWS_COLUMNS) - set(news_df.columns)

    if missing_columns:
        raise ValueError(f'필수 컬럼이 누락되었습니다. {sorted(missing_columns)}')

    required_columns = [
        'rank',
        'search_keyword',
        'press',
        'title',
        'article_url',
        'source_url',
        'collected_at',
    ]

    if news_df[required_columns].isna().any().any():
        raise ValueError('필수 데이터에 결측값이 존재합니다.')

    if news_df['article_url'].duplicated().any():
        raise ValueError('중복된 기사 URL이 존재합니다.')


def save_news_csv(
    news_df: pd.DataFrame,
    csv_file: Path,
) -> Path:
    """뉴스 DataFrame을 CSV로 원자적으로 저장한다."""

    validate_news_dataframe(news_df)

    csv_file.parent.mkdir(parents=True, exist_ok=True)

    temp_file = csv_file.with_suffix('.tmp')

    news_df[NEWS_COLUMNS].to_csv(
        temp_file,
        index=False,
        encoding='utf-8-sig',
    )

    temp_file.replace(csv_file)

    return csv_file


def verify_saved_csv(
    csv_file: Path,
    original_df: pd.DataFrame,
) -> pd.DataFrame:
    """저장한 CSV를 다시 읽어 행 수와 기사 URL을 검증한다."""

    saved_df = pd.read_csv(csv_file)

    if len(saved_df) != len(original_df):
        raise ValueError('CSV 저장 전후의 행 수가 다릅니다.')

    if saved_df['article_url'].tolist() != original_df['article_url'].tolist():
        raise ValueError('CSV 저장 전후의 기사 URL 순서가 다릅니다.')

    return saved_df

# 전체 실행 함수

실무 흐름상 **이미지 다운로드 결과까지 CSV 메타데이터에 기록하기 위해**
다음 순서로 실행한다.

```text
네이버 뉴스 검색
→ 현재 페이지 기사 추출
→ 이미지 다운로드
→ 이미지 저장 결과를 메타데이터에 추가
→ CSV 저장
→ CSV 재검증
```


In [29]:
def run_naver_news_collection(
    search_keyword: str = SEARCH_KEYWORD,
    headless: bool = HEADLESS,
    wait_timeout: int = WAIT_TIMEOUT,
) -> tuple[pd.DataFrame, Path, Path]:
    """
    네이버 뉴스 검색부터 현재 페이지 기사·이미지 저장까지 실행한다.

    Returns:
        1. 최종 뉴스 DataFrame
        2. CSV 파일 경로
        3. 이미지 저장 폴더 경로
    """

    collected_at = datetime.now()

    batch_dir, image_dir = create_batch_directory(
        output_dir=OUTPUT_DIR,
        search_keyword=search_keyword,
        collected_at=collected_at,
    )

    batch_name = collected_at.strftime('%Y%m%d_%H%M%S')
    keyword_name = sanitize_name(search_keyword)
    csv_file = batch_dir / f'naver_news_{keyword_name}_{batch_name}.csv'

    driver = create_driver(headless)
    news_df = None

    try:
        source_url = open_naver_news_search(
            driver=driver,
            search_keyword=search_keyword,
            wait_timeout=wait_timeout,
        )

        print(f'검색 결과 URL : {source_url}')

        news_df = collect_current_news_page(
            driver=driver,
            search_keyword=search_keyword,
            collected_at=collected_at,
            wait_timeout=wait_timeout,
        )

        news_df = download_news_images(
            news_df=news_df,
            image_dir=image_dir,
            project_dir=PROJECT_DIR,
        )

        save_news_csv(
            news_df=news_df,
            csv_file=csv_file,
        )

        verify_saved_csv(
            csv_file=csv_file,
            original_df=news_df,
        )

    finally:
        driver.quit()

    downloaded_count = int(news_df['image_downloaded'].sum())

    print()
    print('=' * 60)
    print('네이버 뉴스 수집 완료')
    print('=' * 60)
    print(f'검색어 : {search_keyword}')
    print(f'기사 수 : {len(news_df)}')
    print(f'이미지 저장 수 : {downloaded_count}')
    print(f'CSV 파일 : {csv_file}')
    print(f'이미지 폴더 : {image_dir}')

    return news_df, csv_file, image_dir

# 실행

In [30]:
news_df, csv_file, image_dir = run_naver_news_collection(
    search_keyword=SEARCH_KEYWORD,
    headless=HEADLESS,
)

검색 결과 URL : https://search.naver.com/search.naver?ssc=tab.news.all&where=news&sm=tab_jum&query=AI
[004] 기사 추출 실패 : NoSuchElementException
현재 페이지 수집 기사 수 : 9

네이버 뉴스 수집 완료
검색어 : AI
기사 수 : 9
이미지 저장 수 : 9
CSV 파일 : D:\AI\data_analytics\crawling\01-data-collection-pipeline\data\dynamic\naver\AI\20260818_094819\naver_news_AI_20260818_094819.csv
이미지 폴더 : D:\AI\data_analytics\crawling\01-data-collection-pipeline\data\dynamic\naver\AI\20260818_094819\images


# 결과 확인

In [15]:
news_df.head()

,rank,search_keyword,press,title,summary,article_url,image_url,image_source_url,source_site,source_url,collected_at,image_file,image_downloaded
0,1,AI,연합뉴스,"엔비디아, 오픈AI 데이터센터에 150조원 보증…순환금융 지적도","SB에너지에도 2.1조원 투자…젠슨 황 ""오픈AI에서만 850조원 매출"" 권영전 특...",https://www.yna.co.kr/view/AKR2026081800490009...,https://search.pstatic.net/common/?src=https%3...,https://imgnews.pstatic.net/image/origin/001/2...,Naver News Search,https://search.naver.com/search.naver?ssc=tab....,2026-08-18T09:25:32,data\dynamic\naver\AI\20260818_092532\images\0...,True
1,2,AI,뉴스1,"청년 2명 중 1명 ""AI가 내 직무 5년 안에 대체""…미취업 64% '불안'",인공지능(AI)에 의해 대체되거나 축소될 수 있다고 우려하는 것으로 나타났다. 구직...,https://www.news1.kr/industry/general-industry...,https://search.pstatic.net/common/?src=https%3...,https://imgnews.pstatic.net/image/origin/421/2...,Naver News Search,https://search.naver.com/search.naver?ssc=tab....,2026-08-18T09:25:32,data\dynamic\naver\AI\20260818_092532\images\0...,True
2,3,AI,SBS,"할리우드 영화계, AI 기업과 사상 첫 '저작권 보호' 합의",▲ 아일랜드 출신 영화 감독 루어리 로빈슨이 시댄스로 생성한 브래드 피트와 톰 크루...,https://news.sbs.co.kr/news/endPage.do?news_id...,https://search.pstatic.net/common/?src=https%3...,https://imgnews.pstatic.net/image/origin/055/2...,Naver News Search,https://search.naver.com/search.naver?ssc=tab....,2026-08-18T09:25:32,data\dynamic\naver\AI\20260818_092532\images\0...,True
3,4,AI,뉴시스,"""자격증 몰라도 AI에 물어보세요""…시험 정보 1분 만에 답변",앞으로는 국가자격시험의 정확한 종목명을 몰라도 인공지능(AI)을 통해 자격 정보를 ...,https://www.newsis.com/view/NISX20260817_00037...,https://search.pstatic.net/common/?src=https%3...,https://imgnews.pstatic.net/image/origin/003/2...,Naver News Search,https://search.naver.com/search.naver?ssc=tab....,2026-08-18T09:25:32,data\dynamic\naver\AI\20260818_092532\images\0...,True
4,6,AI,동아일보,"바다 지키다 순직한 해양경찰관, AI로 복원돼 깊은 울림",우리 영해에서 불법 중국어선을 단속하고 국민 생명을 지키다 순직한 해양경찰관들의 모...,https://www.donga.com/news/Society/article/all...,https://search.pstatic.net/common/?src=https%3...,https://imgnews.pstatic.net/image/origin/020/2...,Naver News Search,https://search.naver.com/search.naver?ssc=tab....,2026-08-18T09:25:32,data\dynamic\naver\AI\20260818_092532\images\0...,True


In [16]:
print(f'수집 기사 수 : {len(news_df)}')
print(f'이미지 저장 수 : {int(news_df["image_downloaded"].sum())}')
print(f'CSV 파일 : {csv_file}')
print(f'이미지 폴더 : {image_dir}')

수집 기사 수 : 9
이미지 저장 수 : 9
CSV 파일 : D:\AI\data_analytics\crawling\01-data-collection-pipeline\data\dynamic\naver\AI\20260818_092532\naver_news_AI_20260818_092532.csv
이미지 폴더 : D:\AI\data_analytics\crawling\01-data-collection-pipeline\data\dynamic\naver\AI\20260818_092532\images


# 최종 저장 구조

```text
data/
└─ dynamic/
   └─ naver/
      └─ AI/
         └─ YYYYMMDD_HHMMSS/
            ├─ images/
            │  ├─ 001_기사제목.jpg
            │  ├─ 002_기사제목.jpg
            │  └─ ...
            └─ naver_news_AI_YYYYMMDD_HHMMSS.csv
```

### 함수 흐름

```text
run_naver_news_collection()
│
├─ create_batch_directory()
├─ create_driver()
├─ open_naver_news_search()
├─ collect_current_news_page()
│  └─ parse_news_item()
│     └─ get_original_image_url()
├─ download_news_images()
│  ├─ create_http_session()
│  └─ download_image()
│     ├─ get_image_extension()
│     └─ sanitize_name()
├─ save_news_csv()
│  └─ validate_news_dataframe()
└─ verify_saved_csv()
```
